In [33]:
from plots_utils.loading import ExperimentConfig, load_experiment_results
from plots_utils.plot_availability_comparison import plot_availability_comparison
from plots_utils.plot_biased_unbiased_comparison import plot_biased_unbiased_comparison
from plots_utils.plot_av_mat import plot_av_mat
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

from matplotlib import pyplot as plt
# from plots_utils.loading import parse_tf_events_file
import numpy as np
import pandas as pd

from pathlib import Path
import os

def parse_tf_events_file(events_path, tag, time_horizon=None):
    """
    Returns the data in the file located in the folder events_paths,
    and corresponding to the tag.
    The tag can be: 'Train/Loss', 'Train/Metric', 'Test/Loss', 'Test/Metric'.
    """
    ea = EventAccumulator(events_path).Reload()
    # print(list(ea.Scalars(tag)))
    tag_values, steps = [], []
    for event in ea.Scalars(tag):
        if time_horizon is None or event.step <= time_horizon:
            tag_values.append(event.value)
            steps.append(event.step)
    return steps, tag_values

def get_exp_stats(config):
    results = list()
    for lr in config.lr_list:
        # time_horizon = time_horizons[p]  
        for algorithm in config.algorithms:
            # b_loop = config.b_values if algorithm == 'mixture' else [None] # in case we vary beta
            # for b in b_loop:
            for event in config.events:
                for seed in config.seeds:
                    for a in config.alphas:
                        for n_c in config.n_clients_list:
                            for av in config.availabilities:
                                for part in config.participations:
                                    for biased in config.biased_list:

                                        event_dir = config.get_event_dir(algorithm, lr, seed, 
                                                                            event, a, n_c, av, 
                                                                            config.n_rounds, part, biased, config.train_test)
                                        # print(event_dir)


                                        if os.path.exists(event_dir):
                                            # print('x')
                                            # _, values = parse_tf_events_file(event_dir, tag="Test/Metric", time_horizon=time_horizon)
                                            _, test_accuracy_values = parse_tf_events_file(event_dir, tag="Test/Metric")
                                            _, test_loss_values = parse_tf_events_file(event_dir, tag="Test/Loss")
                                            _, train_accuracy_values = parse_tf_events_file(event_dir, tag="Train/Metric")
                                            _, train_loss_values = parse_tf_events_file(event_dir, tag="Train/Loss")
                                            ### tag can be: 'Train/Loss', 'Train/Metric', 'Test/Loss', 'Test/Metric'
                                            max_accuracy = np.array(test_accuracy_values).max() * 100
                                            results.append({
                                                "algorithm": algorithm, 
                                                "availability": av,
                                                "alpha": a, 
                                                "participation": part,
                                                "max_test_accuracy": float(max_accuracy),
                                                "final_test_accuracy":test_accuracy_values[-1]*100,
                                                "test_accuracy": np.array(test_accuracy_values),
                                                "seed": seed,
                                                "lr": lr, "event": event, "n_clients": n_c,
                                                "biased": biased
                                            })

                                            # "b": float(b) if b else np.nan # in case we vary beta
    return pd.DataFrame(results)


main_folder = 'mnist_idle_accuracy_check'

availabilities="""
alphaF-13sl-7cb-1ft 
alphaF-13sl-7cb-3ft 
alphaF-14sl-6cb-1ft 
alphaF-14sl-6cb-3ft 
alphaF-15sl-5cb-1ft 
alphaF-15sl-5cb-3ft 
alphaF-19sl-1cb-3ft 
alphaF-20sl-4cb-1ft 
alphaF-20sl-4cb-3ft 
alphaF-23sl-7cb-1ft 
alphaF-23sl-7cb-3ft 
alphaF-24sl-6cb-1ft 
alphaF-24sl-6cb-3ft 
alphaF-25sl-5cb-1ft 
alphaF-25sl-5cb-3ft 
alphaF-29sl-1cb-3ft 
alphaF-30sl-3cb-1ft 
alphaF-30sl-3cb-3ft 
alphaF-30sl-4cb-1ft 
alphaF-30sl-4cb-3ft 
alphaF-34sl-6cb-1ft 
alphaF-34sl-6cb-3ft 
alphaF-35sl-5cb-1ft 
alphaF-35sl-5cb-3ft 
alphaF-39sl-1cb-3ft 
alphaF-40sl-2cb-1ft 
alphaF-40sl-2cb-3ft 
alphaF-40sl-3cb-1ft 
alphaF-40sl-3cb-3ft 
alphaF-40sl-4cb-1ft 
alphaF-40sl-4cb-3ft 
alphaF-45sl-5cb-1ft 
alphaF-45sl-5cb-3ft 
alphaF-49sl-1cb-3ft 
alphaF-50sl-1cb-1ft 
alphaF-50sl-1cb-3ft 
alphaF-50sl-2cb-1ft 
alphaF-50sl-2cb-3ft 
alphaF-50sl-3cb-1ft 
alphaF-50sl-3cb-3ft 
alphaF-50sl-4cb-1ft 
alphaF-50sl-4cb-3ft 
alphaF-59sl-1cb-3ft 
alphaF-60sl-1cb-1ft 
alphaF-60sl-1cb-3ft 
alphaF-60sl-2cb-1ft 
alphaF-60sl-2cb-3ft 
alphaF-60sl-3cb-1ft 
alphaF-60sl-3cb-3ft 
alphaF-60sl-4cb-1ft 
alphaF-60sl-4cb-3ft 
alphaF-70sl-1cb-1ft 
alphaF-70sl-1cb-3ft 
alphaF-70sl-2cb-1ft 
alphaF-70sl-2cb-3ft 
alphaF-70sl-3cb-1ft 
alphaF-70sl-3cb-3ft 
alphaF-79sl-1cb-3ft 
alphaF-80sl-1cb-1ft 
alphaF-80sl-1cb-3ft 
alphaF-80sl-2cb-1ft 
alphaF-80sl-2cb-3ft 
alphaF-80sl-4cb-1ft 
alphaF-80sl-4cb-3ft 
alphaF-90sl-1cb-1ft 
alphaF-90sl-1cb-3ft 
alphaF-90sl-3cb-1ft 
alphaF-90sl-3cb-3ft 
alphaF-100sl-2cb-1ft 
alphaF-100sl-2cb-3ft 
alphaF-110sl-1cb-1ft 
alphaF-110sl-1cb-3ft 
alphaF-110sl-3cb-1ft 
alphaF-110sl-3cb-3ft 
alphaF-120sl-2cb-1ft 
alphaF-120sl-2cb-3ft 
alphaF-130sl-1cb-1ft 
alphaF-130sl-1cb-3ft 
alphaF-130sl-3cb-1ft 
alphaF-130sl-3cb-3ft 
alphaF-140sl-2cb-1ft 
alphaF-140sl-2cb-3ft 
alphaF-150sl-1cb-1ft 
alphaF-150sl-1cb-3ft 
alphaF-150sl-3cb-1ft 
alphaF-150sl-3cb-3ft 
alphaF-160sl-2cb-1ft 
alphaF-160sl-2cb-3ft 
alphaF-170sl-1cb-1ft 
alphaF-170sl-1cb-3ft 
alphaF-180sl-2cb-1ft 
alphaF-180sl-2cb-3ft 
alphaF-190sl-1cb-1ft 
alphaF-190sl-1cb-3ft 
""".split()


config = ExperimentConfig(base_path=os.path.join('..', 'logs'), experiment="mnist_idle0.1", seeds=["42", '78', '84'],
                          algorithms=["fedavg"], events=["global"],
                          lr_list=['5e-2', '1e-2'], alphas=["0.5"], n_clients_list=["7"],
                          availabilities=availabilities,
                          n_rounds="100", participations=["known"], biased_list=["2"], train_test="train")

results_df = get_exp_stats(config)



import ast

# Convert 'test_accuracy' column to lists if needed
results_df['test_accuracy'] = results_df['test_accuracy'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)



# # Loop over availability groups
# for availability_value, group in results_df.groupby("availability"):
    
#     plt.figure(figsize=(10, 6))
    
#     for _, row in group.iterrows():
#         label = f"lr={row['lr']} | seed={row['seed']}"
#         plt.plot(row['test_accuracy'], label=label)
    
#     plt.title(f"Test Accuracy Curves — availability = {availability_value}")
#     plt.xlabel("Epoch")
#     plt.ylabel("Test Accuracy")
#     plt.grid(True)
#     plt.legend()
#     plt.show()



In [36]:
cb_to_time = [40, 30, 20, 10, 9, 5, 4]

results_df = results_df[['availability','final_test_accuracy', 'seed','lr']]
tmp = results_df.groupby(['availability']).final_test_accuracy.apply(np.vstack).to_frame().reset_index()
tmp['var_test_acc'] = tmp['final_test_accuracy'].apply(lambda x : x.var(axis=0))
tmp['max_test_acc'] = tmp['final_test_accuracy'].apply(lambda x: x.max(axis=0))
tmp = tmp[['availability', 'var_test_acc', 'max_test_acc']]
tmp["cb"] = tmp["availability"].str.extract(r"(\d+)cb").astype(int)
tmp["ft"] = tmp["availability"].str.extract(r"(\d+)ft").astype(int)
tmp["sl"] = tmp["availability"].str.extract(r"(\d+)sl").astype(int)
tmp["T"] = tmp["cb"].apply(lambda x : cb_to_time[x-1])
tmp["sl"] = tmp["sl"] - tmp["T"]
tmp[tmp["ft"] == 3]

,availability,var_test_acc,max_test_acc,cb,ft,sl,T
1,alphaF-100sl-2cb-3ft,[0.16961402592367372],[98.97000193595886],2,3,70,30
3,alphaF-110sl-1cb-3ft,[0.12342212907097878],[99.14000034332275],1,3,70,40
5,alphaF-110sl-3cb-3ft,[2.8915134289025306],[97.58999943733215],3,3,90,20
7,alphaF-120sl-2cb-3ft,[0.24329296692530514],[98.79000186920166],2,3,90,30
9,alphaF-130sl-1cb-3ft,[0.19737988755331848],[99.04999732971191],1,3,90,40
11,alphaF-130sl-3cb-3ft,[16.47788709106695],[94.20999884605408],3,3,110,20
13,alphaF-13sl-7cb-3ft,[15.438846415713037],[94.2900002002716],7,3,9,4
15,alphaF-140sl-2cb-3ft,[0.4697899621793529],[97.75999784469604],2,3,110,30
17,alphaF-14sl-6cb-3ft,[14.731682459434325],[94.26000118255615],6,3,9,5
19,alphaF-150sl-1cb-3ft,[0.07280069640655838],[98.79000186920166],1,3,110,40


In [26]:
import pandas as pd
df = tmp
# --- 1) Ensure list-like numeric columns are scalars (e.g., [74.87] -> 74.87) ---
for col in ["var_test_acc", "max_test_acc"]:
    df[col] = df[col].apply(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else x)

# Make sure cb, sl, T are integers (optional but recommended)
df["cb"] = df["cb"].astype(int)
df["sl"] = df["sl"].astype(int)
df["T"]  = df["T"].astype(int)

# --- 2) Pivot to wide: rows are (cb, T), columns are sl, values are max_test_acc ---
wide = (df.pivot_table(index=["cb", "T"], columns="sl", values="max_test_acc", aggfunc="first")
          .sort_index()
          .sort_index(axis=1))

# Optional: nicer LaTeX column header like "sl=80" instead of just "80"
wide.columns = [f"sl={c}" for c in wide.columns]

# --- 3) Export to LaTeX ---
latex_str = wide.to_latex(
    index=True,
    na_rep="",
    float_format="%.2f",
    caption="Max test accuracy by cb, T and sl",
    label="tab:cb_T_sl",
    column_format="ll" + "r"*wide.shape[1]  # 2 left cols (cb,T) + right-aligned numeric cols
)

print(latex_str)


\begin{table}
\caption{Max test accuracy by cb, T and sl}
\label{tab:cb_T_sl}
\begin{tabular}{llrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 &  & sl=21 & sl=30 & sl=41 & sl=50 & sl=55 & sl=60 & sl=61 & sl=65 & sl=70 & sl=75 & sl=80 & sl=81 & sl=85 & sl=90 & sl=95 & sl=100 & sl=101 & sl=105 & sl=110 & sl=115 & sl=120 & sl=121 & sl=125 & sl=130 & sl=135 & sl=140 & sl=145 & sl=150 \\
cb & T &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
1 & 40 &  &  &  &  &  &  &  &  & [74.94999766] &  &  &  &  & [74.87000227] &  &  &  &  & [75.52000284] &  &  &  &  & [75.44000149] &  &  &  & [75.55000186] \\
\cline{1-30}
2 & 30 &  &  &  &  &  &  &  & [74.83000159] &  &  &  &  & [74.91000295] &  &  &  &  & [75.01999736] &  &  &  &  & [74.98999834] &  &  &  & [75.29000044] &  \\
\cline{1-30}
3 & 20 &  &  &  &  &  & [74.29999709] &  &  &  &  & [74.87000227] &  &  &  &  & [74.58000183] &  &  &  &  & [74.81999993] &  &  &  &  & [75.2399981] &  &  \\
\cline{1-30}
4 &